In [ ]:
# Optional: install required libraries for this lecture
%pip install -q openai


### Step 1: Enter your OpenRouter key and set a daily budget (tokens)
We'll track total tokens used today and alert at 80% of the cap.
Your API key is entered securely and is not saved in the notebook.


In [4]:
import os, json, datetime
from getpass import getpass
from openai import OpenAI

OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key: ")

client = OpenAI(
    api_key=OPENROUTER_API_KEY,
    base_url="https://openrouter.ai/api/v1",
)

MODEL = "openai/gpt-4o-mini"  # Change to any OpenRouter-supported model

DAILY_BUDGET_TOKENS = 2000  # change for your needs
ALERT_THRESHOLD = 0.8
LOG_PATH = "usage_log.json"

def today_key():
    return datetime.date.today().isoformat()

# Initialize log
if not os.path.exists(LOG_PATH):
    with open(LOG_PATH, 'w') as f:
        json.dump({}, f)

with open(LOG_PATH, 'r') as f:
    log = json.load(f)

today = today_key()
if today not in log:
    log[today] = {"prompt_tokens": 0, "completion_tokens": 0}


### Step 2: Wrap a chat call and log tokens
We use the API's usage fields to track tokens and write back to file.


In [5]:
def chat_and_log(prompt: str):
    resp = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": prompt}],
        temperature=0,
    )

    usage = resp.usage

    # Update today's log
    log[today]["prompt_tokens"] += getattr(usage, "prompt_tokens", 0) or 0
    log[today]["completion_tokens"] += getattr(usage, "completion_tokens", 0) or 0

    with open(LOG_PATH, 'w') as f:
        json.dump(log, f, indent=2)

    total = log[today]["prompt_tokens"] + log[today]["completion_tokens"]
    pct = total / DAILY_BUDGET_TOKENS
    print(f"Used today: {total} tokens ({pct:.0%} of budget)")

    if pct >= ALERT_THRESHOLD:
        print("ALERT: You reached 80% of today's token budget.")

    return resp.choices[0].message.content


### Step 3: Try a couple of calls
Small prompts keep demos fast and cheap.


In [7]:
print(chat_and_log("Give one 5-word greeting."))
print()
print(chat_and_log("Name two fruits."))


Used today: 95 tokens (5% of budget)
Hello! Hope you're having fun!

Used today: 110 tokens (6% of budget)
Apple and banana.
